In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_excel("../data/FINAL_ITO_DATA.xlsx")

# Creating a proper date column (Why? Because Python can then understand that January 2 comes after January 1, calculate time differences, extract weekdays and create historical features.)
df["date"] = pd.to_datetime({
    "year": df["Year"],
    "month": df["Month"],
    "day": df["Date"]
})

# Sorting the data by date (Why? Because we want to make sure that the data is in chronological order, which is important for time series analysis.)
df = df.sort_values("date").reset_index(drop=True)

print(df[["date", "AQI"]].head(10))

        date    AQI
0 2021-01-01  432.0
1 2021-01-02  418.0
2 2021-01-03  306.0
3 2021-01-04  138.0
4 2021-01-05  129.0
5 2021-01-06  196.0
6 2021-01-07  191.0
7 2021-01-08  170.0
8 2021-01-09  285.0
9 2021-01-10  210.0


In [7]:
# Check the first and last few rows of the dataset
print("Start date:", df["date"].min())
print("End date:", df["date"].max())

# Check duplicate dates
print("Duplicate dates:", df["date"].duplicated().sum())

# Calculating differences between consecutive dates to find missing dates
df["date_diff"] = df["date"].diff()

print(df["date_diff"].value_counts())

Start date: 2021-01-01 00:00:00
End date: 2024-12-31 00:00:00
Duplicate dates: 0
date_diff
1 days    1460
Name: count, dtype: int64


In [5]:
# Pandas weekday: Monday=0, Sunday=6
df["weekday_check"] = df["date"].dt.dayofweek + 1

print((df["Days"] == df["weekday_check"]).all())

True


In [23]:
pollutants = ["PM2.5", "PM10", "NO2", "SO2", "CO", "Ozone"]

# Checking negative values 
print("Negative pollutant values:")
print((df[pollutants]<0).sum())

# Checking AQI range
print("Minimum AQI:", df["AQI"].min())
print("Maximum AQI:", df["AQI"].max())

#Checking the invalid AQI values
print(
    ((df["AQI"] < 0) | (df["AQI"] > 500 )).sum()
)

Negative pollutant values:
PM2.5    0
PM10     0
NO2      0
SO2      0
CO       0
Ozone    0
dtype: int64
Minimum AQI: 37.0
Maximum AQI: 500.0
0


In [24]:
# keeping original columns and new date column
df_clean = df.drop(
    columns = ["date_diff", "weekday_check"], 
    errors = "ignore"
).copy()

# Verifying the final dataset
print("Shape:", df_clean.shape)

print("Missing values:")
print(df_clean.isnull().sum().sum())

print("Duplicate rows:", df_clean.duplicated().sum())

print(df_clean.head()) 

Shape: (1461, 13)
Missing values:
0
Duplicate rows: 0
   Date  Month  Year  Holidays_Count  Days   PM2.5    PM10    NO2    SO2  \
0     1      1  2021               0     5  363.12  459.14  13.24  21.39   
1     2      1  2021               0     6  260.94  325.94  12.76  20.41   
2     3      1  2021               1     7  117.07  144.22  11.25  12.43   
3     4      1  2021               0     1   80.38  109.95  14.40  11.05   
4     5      1  2021               0     2   60.66   83.23  13.28  11.22   

     CO  Ozone    AQI       date  
0  6.75   8.54  432.0 2021-01-01  
1  4.45   6.77  418.0 2021-01-02  
2  3.13   6.04  306.0 2021-01-03  
3  3.00   6.78  138.0 2021-01-04  
4  2.75   6.33  129.0 2021-01-05  


In [25]:
df_clean.to_csv(
    "../data/Delhi_AQI_Cleaned.csv",
    index = False
)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!
